# Online Retail Sales — ETL and Data Analysis Pipeline

**Current progress checkpoint:** Dataset setup, extraction, and initial exploratory data analysis (EDA).

## Project objective
Explore online retail transactions to understand sales patterns across products, time, and countries, then build a reproducible ETL pipeline that loads cleaned data into a relational database and supports SQL reporting.

## Dataset
- **Name:** Online Retail
- **Source:** UCI Machine Learning Repository
- **Source page:** https://archive.ics.uci.edu/dataset/352/online+retail
- **Original format:** Excel workbook
- **Important:** Keep the downloaded original workbook unchanged in `data/raw/`.

The UCI repository describes this as a transaction dataset from a UK-based online retailer covering December 2010 to December 2011. The dataset is licensed under CC BY 4.0; cite the source in the final report.

## Planned pipeline
1. Extract raw data
2. Explore dataset and check data quality
3. Clean and transform
4. Answer at least five analytical questions
5. Design related relational tables and an ERD
6. Load data into a relational database
7. Write at least eight SQL reporting queries
8. Document results and demonstrate the pipeline

## 1. Setup

Download `Online Retail.xlsx` from the UCI source page and save it to `data/raw/Online Retail.xlsx`. Do not edit the raw file.

Install the packages listed in `requirements.txt` if they are not already installed.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
RAW_FILE = PROJECT_DIR / "data" / "raw" / "Online Retail.xlsx"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Raw dataset found:", RAW_FILE.exists())

if not RAW_FILE.exists():
    raise FileNotFoundError(
        "Please download 'Online Retail.xlsx' from "
        "https://archive.ics.uci.edu/dataset/352/online+retail "
        "and place it in data/raw/."
    )

## 2. Extract the raw dataset

Read the source workbook with Pandas. The source file is preserved; this notebook does not overwrite it.

In [ ]:
raw_df = pd.read_excel(RAW_FILE)

print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])
display(raw_df.head())

## 3. Basic dataset exploration

Inspect the schema, data types, sample records, summary statistics, and distinct values.

In [ ]:
print("Dataset shape:", raw_df.shape)
print("\\nColumn names:")
print(raw_df.columns.tolist())

print("\\nData types:")
display(raw_df.dtypes.rename("data_type").to_frame())

print("\\nSummary statistics (including categorical columns):")
display(raw_df.describe(include="all").T)

print("\\nUnique values per column:")
display(raw_df.nunique(dropna=False).rename("unique_values").to_frame())

print("\\nRandom sample:")
display(raw_df.sample(min(5, len(raw_df)), random_state=42))

## 4. Initial data-quality checks

This section identifies issues to investigate. It does not automatically delete records or fill missing values; cleaning decisions will be documented in the next phase.

In [ ]:
quality_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percent": (raw_df.isna().mean() * 100).round(2),
    "dtype": raw_df.dtypes.astype(str),
    "unique_values": raw_df.nunique(dropna=False)
})
display(quality_summary)

duplicate_count = int(raw_df.duplicated().sum())
print("Exact duplicate rows:", duplicate_count)

# Check numeric ranges and potentially unusual values.
for col in ["Quantity", "UnitPrice"]:
    if col in raw_df.columns:
        print(f"\\n{col} summary:")
        display(raw_df[col].describe())
        print(f"Rows where {col} <= 0:", int((raw_df[col] <= 0).sum()))

# Cancellation invoices are identified by an invoice number beginning with C.
if "InvoiceNo" in raw_df.columns:
    cancellation_mask = raw_df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
    print("\\nRows with cancellation-style invoice numbers:", int(cancellation_mask.sum()))

## 5. Prepare a working copy and explore sales patterns

Create a separate working copy, convert the invoice date, and calculate line-level sales value. Negative quantities and cancellation invoices may represent returns/cancellations, so they are reported separately rather than silently removed at this stage.

In [ ]:
eda_df = raw_df.copy()

eda_df["InvoiceDate"] = pd.to_datetime(eda_df["InvoiceDate"], errors="coerce")
eda_df["LineValue"] = eda_df["Quantity"] * eda_df["UnitPrice"]
eda_df["InvoiceMonth"] = eda_df["InvoiceDate"].dt.to_period("M").astype("string")

cancellation_mask = eda_df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
print("Rows in working copy:", len(eda_df))
print("Rows with missing customer IDs:", int(eda_df["CustomerID"].isna().sum()))
print("Cancellation-style invoice rows:", int(cancellation_mask.sum()))
print("Rows with non-positive quantity:", int((eda_df["Quantity"] <= 0).sum()))
print("Rows with non-positive unit price:", int((eda_df["UnitPrice"] <= 0).sum()))

print("\\nTop countries by transaction-row count:")
display(eda_df["Country"].value_counts().head(10).rename_axis("Country").to_frame("Rows"))

print("\\nTop products by quantity (including all recorded quantities):")
display(
    eda_df.groupby("Description", dropna=False)["Quantity"]
    .sum().sort_values(ascending=False).head(10).to_frame("TotalQuantity")
)

print("\\nMonthly recorded line value (includes negative/return lines where present):")
monthly_value = eda_df.groupby("InvoiceMonth", dropna=False)["LineValue"].sum().sort_index()
display(monthly_value.to_frame("RecordedLineValue"))

## 6. Initial visualizations

These charts are exploratory. Interpret sales totals carefully until return/cancellation handling is decided during cleaning.

In [ ]:
# Monthly recorded line value
monthly_plot = monthly_value.dropna()
if not monthly_plot.empty:
    ax = monthly_plot.plot(kind="line", marker="o", figsize=(10, 4), title="Monthly Recorded Line Value")
    ax.set_xlabel("Invoice month")
    ax.set_ylabel("Line value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Top 10 countries by row count
country_counts = eda_df["Country"].value_counts().head(10).sort_values()
ax = country_counts.plot(kind="barh", figsize=(8, 5), title="Top 10 Countries by Transaction Rows")
ax.set_xlabel("Rows")
ax.set_ylabel("Country")
plt.tight_layout()
plt.show()

## 7. Save initial EDA outputs

Save compact data-quality and monthly summaries so the findings can be included in the progress update and final report.

In [ ]:
quality_summary.to_csv(OUTPUT_DIR / "initial_data_quality_summary.csv", index=True)
monthly_value.rename("recorded_line_value").to_csv(OUTPUT_DIR / "monthly_recorded_line_value.csv")
print("Saved initial EDA outputs to:", OUTPUT_DIR.resolve())

## Progress checkpoint and next steps

### Completed in this notebook
- Project objective and dataset source documented.
- Code prepared to extract the original workbook without overwriting it.
- Initial EDA covers dimensions, schema, summary statistics, unique values, missing values, duplicates, numeric ranges, cancellation-style invoices, country counts, product quantities, and monthly recorded line value.
- Initial charts and CSV summaries are included.

### Still to do
- Run the notebook and review the actual outputs.
- Decide and document cleaning rules based on observed quality issues.
- Build the final cleaned/transformed dataset and answer at least five analytical questions.
- Design related relational database tables and an ERD.
- Load data through Python and produce at least eight SQL reports.
- Complete the project report and final demo.

### Dataset citation
Chen, D. (2015). *Online Retail* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33